In [0]:
import os
import pyspark.sql.functions as F
from pyspark.sql import DataFrame, SparkSession

def initialize_environment(spark: SparkSession, active_catalog: str, schemas: list[str]):
    """Ensures catalog schemas exist in Unity Catalog."""
    for schema in schemas:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {active_catalog}.{schema}")
        full_path = f"{active_catalog}.{schema}"
        assert spark.catalog.databaseExists(full_path), f"INIT FAILED: Schema '{full_path}' not found."


def sanitize_bronze_strings(df: DataFrame) -> DataFrame:
    """Purges '\\N', 'NULL', 'null', and empty whitespace across all string columns."""
    string_cols = [f.name for f in df.schema.fields if f.dataType.typeName() == "string"]
    df_clean = df
    for col_name in string_cols:
        df_clean = df_clean.withColumn(
            col_name,
            F.when(
                (F.col(col_name) == r"\N") | 
                (F.lower(F.col(col_name)) == "null") |
                (F.trim(F.col(col_name)) == ""), 
                None
            ).otherwise(F.col(col_name))
        )
    return df_clean


def run_circuit_breaker(df: DataFrame, primary_keys: list[str], prohibited_pii: set[str] = None):
    """
    Enforces non-null primary keys, key uniqueness, and strict PII exclusion.
    Halts execution before storage is touched if assertions fail.
    """
    if prohibited_pii is None:
        prohibited_pii = {"dob", "url"}

    # 1. Primary Key Non-Null Assertion
    for pk in primary_keys:
        null_count = df.filter(F.col(pk).isNull()).count()
        assert null_count == 0, f"CIRCUIT BREAKER: Found {null_count} NULL values in primary key '{pk}'!"

    # 2. Key Uniqueness Assertion
    total_count = df.count()
    distinct_count = df.select(*primary_keys).distinct().count()
    assert total_count == distinct_count, \
        f"CIRCUIT BREAKER: Key collision detected on {primary_keys}! Total: {total_count}, Unique: {distinct_count}"

    # 3. PII Compliance Assertion
    detected_pii = prohibited_pii.intersection(set(df.columns))
    assert len(detected_pii) == 0, f"CIRCUIT BREAKER: Prohibited PII columns detected: {detected_pii}"

    print(f"✓ Circuit Breaker Passed: {total_count} rows validated on keys {primary_keys}.")


def upsert_silver_delta(spark: SparkSession, df: DataFrame, target_table: str, primary_keys: list[str]):
    """Registers staging view, guarantees CDF & auto-optimize properties, and executes SCD Type 1 MERGE."""
    table_name_short = target_table.split(".")[-1]
    view_name = f"stg_{table_name_short}"
    df.createOrReplaceTempView(view_name)

    # Enable CDF & Auto-Optimize
    spark.sql(f"""
        ALTER TABLE {target_table} SET TBLPROPERTIES (
            'delta.enableChangeDataFeed' = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
    """)

    # Build dynamic join condition: target.pk1 = source.pk1 AND target.pk2 = source.pk2
    join_condition = " AND ".join([f"target.{pk} = source.{pk}" for pk in primary_keys])

    # Execute dynamic MERGE
    spark.sql(f"""
        MERGE INTO {target_table} AS target
        USING {view_name} AS source
        ON {join_condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✓ Upsert Complete: Executed SCD Type 1 MERGE into '{target_table}'.")


def apply_silver_tags(spark: SparkSession, active_catalog: str, target_table: str, current_user_email: str, table_type: str = "dimension"):
    """Applies Unity Catalog governance tags and asserts successful tag writing."""
    spark.sql(f"""
        ALTER TABLE {target_table} SET TAGS (
            'layer' = 'silver',
            'table_type' = '{table_type}',
            'contains_pii' = 'false_anonymized',
            'last_updated_by' = '{current_user_email}'
        )
    """)
    
    # Assert tags exist in Unity Catalog information schema
    table_name_short = target_table.split(".")[-1]
    tags_df = spark.sql(f"""
        SELECT tag_name, tag_value 
        FROM {active_catalog}.information_schema.table_tags 
        WHERE table_name = '{table_name_short}' AND schema_name = 'silver'
    """)
    tag_dict = {row["tag_name"]: row["tag_value"] for row in tags_df.collect()}
    
    assert tag_dict.get("layer") == "silver", f"TAG ERROR: Expected layer='silver', got '{tag_dict.get('layer')}'"
    assert tag_dict.get("last_updated_by") == current_user_email, "TAG ERROR: Audit email mismatch"
    print(f"✓ Governance Tags Verified for '{target_table}'.")